# 08 Demo app

CSE 4889 Machine Learning: Explainable Multi-Label Hate Speech Detection in Transliterated Bangla

A web app for the project show. You type or pick a Banglish comment, and the app shows:
- whether it is hate speech, with the model's confidence,
- which of the 8 hate categories apply,
- which words led to the decision, using SHAP or LIME.

It runs from Colab with a public link (share=True), so it also opens on phones. The link only works while this notebook is running, and it expires after 72 hours.

Runtime: T4 GPU (explanations take about 1 to 3 seconds on GPU and much longer on CPU).

## Install LIME

Gradio and SHAP are already in Colab.

In [1]:
!pip install -q lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 13.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports and paths

The model and the LIME/SHAP code come from src/, the same code used in the explanation notebooks.

In [3]:
import os
import re
import sys
import json
import html
import time

import numpy as np
import torch
import gradio as gr
from transformers import AutoTokenizer

BASE = '/content/drive/MyDrive/ML_Project'
SEED = 42
REP_DIR = os.path.join(BASE, 'results', 'evaluation_reports')
CKPT_DIR = os.path.join(BASE, 'results', 'model_checkpoints')
sys.path.append(os.path.join(BASE, 'src'))
from model import load_trained, CATEGORIES
from explainability.explain import make_predict_fn, lime_word_weights, shap_word_weights

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| Gradio', gr.__version__)

Device: cuda | Gradio 6.26.0


## Load the trained model and its thresholds

We load best_model.pt and model_config.json from the training notebook. The decision thresholds are the ones tuned on the validation set, so the app decides exactly like the model in our results tables.

On CPU we use fewer LIME and SHAP samples so the app stays usable.

In [4]:
with open(os.path.join(CKPT_DIR, 'model_config.json')) as f:
    cfg = json.load(f)
tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])
model = load_trained(cfg['model_name'], os.path.join(CKPT_DIR, cfg['weights_file']), device)
predict_hate = make_predict_fn(model, tokenizer, cfg['max_length'], device)

BIN_THR = cfg['binary_threshold']
CAT_THR = cfg['category_thresholds']
LIME_SAMPLES, SHAP_EVALS = (1000, 500) if device.type == 'cuda' else (300, 200)
print('Binary threshold:', BIN_THR)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Binary threshold: 0.58


## Prediction and cleaning

We clean the input the same way as the training data (remove URLs and extra spaces), then get the hate probability and the 8 category probabilities.

In [5]:
URL_RE = re.compile(r'http\S+|www\S+')
SPACE_RE = re.compile(r'\s+')

def clean_text(t):
    return SPACE_RE.sub(' ', URL_RE.sub('', str(t))).strip()

@torch.no_grad()
def predict_full(text):
    enc = tokenizer([text], truncation=True, max_length=cfg['max_length'], return_tensors='pt').to(device)
    b, c = model(enc['input_ids'], enc['attention_mask'])
    return float(torch.sigmoid(b)[0]), torch.sigmoid(c)[0].cpu().numpy()

print(predict_full('Khankir polar kaner niche ekta marte parle'))

(0.9765129089355469, array([0.15836157, 0.042003  , 0.10757374, 0.7052636 , 0.874538  ,
       0.02178194, 0.06898952, 0.07558977], dtype=float32))


## Key numbers for the "How it works" section

These numbers are read from our saved results files, not typed in by hand, so the app always matches the report.

In [6]:
def load_json(name):
    with open(os.path.join(REP_DIR, name), encoding='utf-8') as f:
        return json.load(f)

m_data = load_json('metrics_data.json')
m_train = load_json('metrics_training.json')
m_rat = load_json('metrics_rationale.json')
m_bias = load_json('metrics_bias.json')

tuned = next(r for r in m_train['test_binary'] if r['Threshold'] == 'Tuned on validation')
shap_plaus = next(r for r in m_rat['plausibility'] if r['Reference'] == 'Annotator A' and r['Method'] == 'SHAP')
rand_plaus = next(r for r in m_rat['plausibility'] if r['Reference'] == 'Annotator A' and r['Method'] == 'Random')
shap_faith = next(r for r in m_rat['faithfulness'] if r['Method'] == 'SHAP')
rand_faith = next(r for r in m_rat['faithfulness'] if r['Method'] == 'Random')
sizes = m_data['split_sizes']
bias_a = next(r for r in m_bias['summary'] if r['Group'].startswith('A'))
bias_b = next(r for r in m_bias['summary'] if r['Group'].startswith('B'))

ABOUT = f"""
**Model.** XLM-RoBERTa (base), fine-tuned with two outputs: hate or not, and 8 hate categories.

**Data.** BanTH, YouTube comments in transliterated Bangla: {sizes['train']:,} training, {sizes['validation']:,} validation and {sizes['test']:,} test comments.

**Test results.** Binary macro-F1 {tuned['Macro-F1']:.1f}% on the official test split.

**Explanations.** SHAP and LIME score every word. Compared with words marked by two human annotators (agreement Kappa {m_rat['cohen_kappa_word_level']:.2f}), SHAP reaches token F1 {shap_plaus['Token F1']:.2f} (random words: {rand_plaus['Token F1']:.2f}). Deleting SHAP's top words lowers the hate probability by {shap_faith['Comp. (k = human)']:.2f} on average (random words: {rand_faith['Comp. (k = human)']:.2f}).

**Known weakness.** In our keyword bias test, {bias_a['False positive rate (%)']:.0f}% of harmless sentences containing words that are common in insults (for example kukur, baccha, pagol, mohila) were wrongly flagged as hate, against {bias_b['False positive rate (%)']:.0f}% of harmless sentences without such words. SHAP named that word as the main reason in {bias_a['Risky word is SHAP top word (%)']:.0f}% of these sentences, so the explanations make the problem visible.

**Please note.** This is a research prototype. It can be wrong, especially for rare categories such as Religious, Origin and Body Shaming. It is meant to help human moderators, not to remove comments automatically.
"""
print(ABOUT)


**Model.** XLM-RoBERTa (base), fine-tuned with two outputs: hate or not, and 8 hate categories.

**Data.** BanTH, YouTube comments in transliterated Bangla: 29,879 training, 3,736 validation and 3,735 test comments.

**Test results.** Binary macro-F1 76.2% on the official test split.

**Explanations.** SHAP and LIME score every word. Compared with words marked by two human annotators (agreement Kappa 0.56), SHAP reaches token F1 0.71 (random words: 0.52). Deleting SHAP's top words lowers the hate probability by 0.48 on average (random words: 0.16).

**Known weakness.** In our keyword bias test, 26% of harmless sentences containing words that are common in insults (for example kukur, baccha, pagol, mohila) were wrongly flagged as hate, against 3% of harmless sentences without such words. SHAP named that word as the main reason in 55% of these sentences, so the explanations make the problem visible.

**Please note.** This is a research prototype. It can be wrong, especially for rare cat

## Example comments

There are two example lists:
- Try an example: the 8 test comments from the explanation notebook (picked by a fixed rule, not by hand) and one clearly positive comment.
- Known weakness: harmless sentences from our keyword bias test that the model wrongly flags as hate. They are picked by a rule too: from the flagged sentences we take the one with the highest P(hate) for each risky word, and keep the top 6. A short note above the list gives the result of the bias test, read from metrics_bias.json.

In [7]:
ex = load_json('explanations_examples.json')
EXAMPLES = [r['text'] for r in ex['comments']]
EXAMPLES += ['apnar video gulo onek sundor hoy, onek kichu shikhlam']

flagged = [r for r in m_bias['sentences'] if r['group'] == 'A' and int(r['flagged']) == 1]
best = {}
for r in sorted(flagged, key=lambda r: -float(r['p_hate'])):
    best.setdefault(r['word'], r)
BIAS_EXAMPLES = [r['sentence'] for r in list(best.values())[:6]]

BIAS_NOTE = (f'<div class="bias-note"><b>Known weakness.</b> These sentences are harmless, but the model flags them as hate '
             f'because they contain words that are common in insults. In our test, '
             f"<b>{bias_a['False positive rate (%)']:.0f}%</b> of harmless sentences with such words were flagged, "
             f"against <b>{bias_b['False positive rate (%)']:.0f}%</b> without them. Click one and look at the red words.</div>")

for e in EXAMPLES:
    print('-', e)
print('Known weakness examples:')
for e in BIAS_EXAMPLES:
    print('-', e)

- Sontrrasi busted durniti bajj khoni president Chuppu chokranto karri sontrrasi busted awamileaq der moddod
- dalalier Krishna Dalali Krishna
- Egular theke dhaka sohorer magirao smart ase koek ta turm mukhostho kore ese vabse dr hoice
- Khankir polar kaner niche ekta marte parle police giri dekhaitam. Baler bacca
- Akhon bachar somoy aise. Ar vab chodau. Gopppo thamau tomra. Sobar khobor ase
- Tui kothay salar bata dasa sso
- Aho vatija.. Samne aho.... tumi to ekta... Futfute sundon... Tore hate paile .. Tor porone kintu jainga-o thakbe na....
- Marbe kano apnader desh e to low ache
- apnar video gulo onek sundor hoy, onek kichu shikhlam
Known weakness examples:
- amar kukurer baccha hoyeche
- oder barite ekta kuttar bacha ache, khub cute
- amar choto baccha school e jay
- tar chul gulo ekdom kalo ar shundor
- amra duijon e boi porar bepare pagol
- mohila club e ajke ekta programme ache


## How the results are shown

These functions build the HTML for the three result panels:
- verdict: a big HATE / NOT HATE card with the probability and a bar with the decision threshold marked,
- categories: one bar per category, where categories above their threshold are marked "detected" (only for comments predicted as hate, like in our evaluation),
- words: every word coloured red (pushes towards hate) or blue (pushes towards non-hate), darker for a bigger effect, the same colours as in the report figures. Hovering over a word shows its exact score.

In [8]:
def verdict_html(p):
    hate = p >= BIN_THR
    cls, label = ('hate', 'Hate speech') if hate else ('ok', 'Not hate speech')
    return f"""
<div class="verdict {cls}">
  <div class="v-top"><span class="v-label">{label}</span><span class="v-prob">{100 * p:.0f}%</span></div>
  <div class="meter"><div class="fill" style="width:{100 * p:.1f}%"></div><div class="thr" style="left:{100 * BIN_THR:.1f}%"></div></div>
  <div class="v-sub">Probability of hate: {p:.2f} &middot; decision threshold {BIN_THR:.2f}</div>
</div>"""

def categories_html(p, cat_probs):
    hate = p >= BIN_THR
    rows = []
    for i in np.argsort(-cat_probs):
        c, q = CATEGORIES[i], float(cat_probs[i])
        on = hate and q >= CAT_THR[c]
        tag = '<span class="tag">detected</span>' if on else '<span></span>'
        rows.append(f'<div class="cat-row {"on" if on else ""}"><span class="cat-name">{c}</span>'
                    f'<div class="cat-track"><div class="cat-fill" style="width:{100 * q:.1f}%"></div></div>'
                    f'<span class="cat-val">{100 * q:.0f}%</span>{tag}</div>')
    note = '' if hate else '<div class="note">Categories are only assigned when a comment is predicted as hate.</div>'
    return f'<div class="panel"><h3>Hate categories</h3>{note}{"".join(rows)}</div>'

def words_html(words, weights, method, seconds):
    m = float(np.abs(weights).max()) or 1.0
    chips = []
    for w, v in zip(words, weights):
        a = 0.12 + 0.78 * abs(v) / m
        rgb = '227,73,72' if v > 0 else '42,120,214'
        chips.append(f'<span class="chip" style="background:rgba({rgb},{a:.2f})" title="{method} score {v:+.3f}">{html.escape(w)}</span>')
    top = [words[i] for i in np.argsort(-weights) if weights[i] > 0][:3]
    top_txt = ', '.join(f'<b>{html.escape(t)}</b>' for t in top) if top else 'none'
    return f"""
<div class="panel why">
  <h3>Why this decision? <span class="method">{method} &middot; {seconds:.1f} s</span></h3>
  <div class="chips">{' '.join(chips)}</div>
  <div class="legend"><span><i class="sw" style="background:rgb(227,73,72)"></i>pushes towards hate</span>
  <span><i class="sw" style="background:rgb(42,120,214)"></i>pushes towards not hate</span>
  <span>darker = stronger</span></div>
  <div class="top">Strongest words towards hate: {top_txt}</div>
</div>"""

EMPTY_VERDICT = '<div class="verdict empty"><div class="v-label">Waiting for a comment</div><div class="v-sub">Type a comment or pick an example, then press Analyze.</div></div>'
EMPTY_WHY = '<div class="panel why empty-why">The highlighted words will appear here.</div>'

## What happens when you press Analyze

analyze() cleans the text, predicts, and explains it with the chosen method. Explanations are cached, so switching back to a method you already used for the same comment is instant. Changing the method only re-runs the explanation, not the prediction.

In [9]:
_cache = {}

def explain(text, method):
    key = (text, method)
    if key not in _cache:
        start = time.time()
        if method == 'LIME':
            words, w = lime_word_weights(text, predict_hate, num_samples=LIME_SAMPLES, seed=SEED)
        else:
            words, w, _ = shap_word_weights(text, predict_hate, max_evals=SHAP_EVALS, seed=SEED)
        _cache[key] = (words, np.asarray(w), time.time() - start)
    return _cache[key]

def analyze(raw, method):
    text = clean_text(raw)
    if not text:
        return EMPTY_VERDICT, '', EMPTY_WHY
    p, cats = predict_full(text)
    words, w, secs = explain(text, method)
    return verdict_html(p), categories_html(p, cats), words_html(words, w, method, secs)

def reexplain(raw, method):
    text = clean_text(raw)
    if not text:
        return EMPTY_WHY
    words, w, secs = explain(text, method)
    return words_html(words, w, method, secs)

_ = analyze(EXAMPLES[0], 'SHAP')
print('Test run OK')

Test run OK


## Look of the app

A calm blue theme with the Inter font, the same red and blue as in the report figures, large text that is readable on a projector, and a layout that also works on a phone. The colours use Gradio's own variables, so the app also looks right in dark mode.

In [10]:
theme = gr.themes.Soft(primary_hue='blue', neutral_hue='slate',
                       font=[gr.themes.GoogleFont('Inter'), 'ui-sans-serif', 'sans-serif'])

CSS = """
.gradio-container {max-width: 1120px !important; margin: 0 auto !important;}
#hdr {padding: 22px 4px 4px;}
#hdr h1 {font-size: 30px; font-weight: 800; margin: 0 0 6px; letter-spacing: -0.02em;}
#hdr p {margin: 0; font-size: 16px; color: var(--body-text-color-subdued); max-width: 760px;}
.badges {margin-top: 12px;}
.badges span {display: inline-block; margin: 0 6px 6px 0; padding: 4px 12px; border-radius: 999px; font-size: 12.5px;
  background: var(--background-fill-secondary); border: 1px solid var(--border-color-primary);}
.panel {border: 1px solid var(--border-color-primary); border-radius: 16px; padding: 16px 18px; background: var(--block-background-fill);}
.panel h3 {margin: 0 0 10px; font-size: 16px; font-weight: 700;}
.verdict {border-radius: 16px; padding: 18px 20px; border: 1.5px solid;}
.verdict.hate {background: rgba(227,73,72,.10); border-color: rgba(227,73,72,.55);}
.verdict.ok {background: rgba(27,175,122,.10); border-color: rgba(27,175,122,.55);}
.verdict.empty {border-style: dashed; border-color: var(--border-color-primary); color: var(--body-text-color-subdued);}
.v-top {display: flex; justify-content: space-between; align-items: baseline; gap: 12px;}
.v-label {font-size: 28px; font-weight: 800; letter-spacing: -0.01em;}
.v-prob {font-size: 28px; font-weight: 700; font-variant-numeric: tabular-nums;}
.verdict.hate .v-label, .verdict.hate .v-prob {color: #d23c3b;}
.verdict.ok .v-label, .verdict.ok .v-prob {color: #14956a;}
.meter {position: relative; height: 10px; border-radius: 99px; background: var(--background-fill-secondary); margin: 12px 0 8px;}
.meter .fill {height: 100%; border-radius: 99px;}
.verdict.hate .fill {background: #e34948;}
.verdict.ok .fill {background: #1baf7a;}
.meter .thr {position: absolute; top: -4px; width: 2px; height: 18px; background: var(--body-text-color);}
.v-sub {font-size: 13px; color: var(--body-text-color-subdued);}
.note {font-size: 13px; color: var(--body-text-color-subdued); margin: -4px 0 8px;}
.cat-row {display: grid; grid-template-columns: minmax(96px, 140px) 1fr 40px 74px; align-items: center; gap: 10px; padding: 4px 0; font-size: 14px;}
.cat-track {height: 10px; border-radius: 99px; background: var(--background-fill-secondary); overflow: hidden;}
.cat-fill {height: 100%; border-radius: 99px; background: #a3acb9;}
.cat-row.on .cat-fill {background: #2a78d6;}
.cat-row.on .cat-name {font-weight: 700;}
.cat-val {text-align: right; font-variant-numeric: tabular-nums; color: var(--body-text-color-subdued);}
.tag {font-size: 11.5px; font-weight: 700; padding: 2px 8px; border-radius: 99px; text-align: center;
  color: #2a78d6; background: rgba(42,120,214,.14);}
.why {margin-top: 2px;}
.panel + .panel {margin-top: 2px;}
.method {font-size: 12.5px; font-weight: 500; color: var(--body-text-color-subdued); margin-left: 8px;}
.chips {font-size: 19px; line-height: 2.5;}
.chip {padding: 5px 9px; border-radius: 9px; white-space: nowrap; color: var(--body-text-color);}
.legend {display: flex; flex-wrap: wrap; gap: 18px; font-size: 13px; color: var(--body-text-color-subdued); margin-top: 8px;}
.sw {display: inline-block; width: 12px; height: 12px; border-radius: 3px; margin-right: 6px; vertical-align: -1px;}
.top {margin-top: 8px; font-size: 14px;}
.empty-why {color: var(--body-text-color-subdued); text-align: center; padding: 26px;}
.bias-note {margin-top: 14px; padding: 12px 14px; border-radius: 12px; font-size: 13.5px; line-height: 1.5;
  background: rgba(235,104,52,.10); border: 1px solid rgba(235,104,52,.45);}
#footer {text-align: center; font-size: 12.5px; color: var(--body-text-color-subdued); padding: 10px 0 2px;}
@media (max-width: 640px) {
  #hdr h1 {font-size: 24px;}
  .v-label, .v-prob {font-size: 22px;}
  .cat-row {grid-template-columns: 92px 1fr 36px 66px; font-size: 13px;}
  .chips {font-size: 16px;}
}
"""

HEADER = """
<div id="hdr">
  <h1>Banglish Hate Speech Detector</h1>
  <p>Detects hate speech in Bangla written in English letters, names the type of hate, and shows which words led to the decision.</p>
  <div class="badges"><span>XLM-RoBERTa</span><span>BanTH dataset</span><span>8 hate categories</span><span>SHAP &amp; LIME explanations</span></div>
</div>"""
FOOTER = '<div id="footer">CSE 4889 Machine Learning &middot; United International University &middot; research prototype</div>'

## Build the layout

Left: the comment box, the explanation method switch, the Analyze button, the example comments and the known-weakness examples. Right: the verdict, the highlighted words and the categories, so the whole result is visible without scrolling. At the bottom there is a "How it works" section that can be opened.

Clicking an example fills the box and analyzes it straight away, with the method that is currently selected.

Newer Gradio versions (6 and up) take the theme and CSS in launch() instead of Blocks(), so we check the version.

In [11]:
GR6 = int(gr.__version__.split('.')[0]) >= 6
style = {'theme': theme, 'css': CSS}

def make_picker(pool):
    def pick(sample, method):
        text = sample[0] if isinstance(sample, (list, tuple)) else pool[int(sample)]
        return (text, *analyze(text, method))
    return pick

with gr.Blocks(title='Banglish Hate Speech Detector', **({} if GR6 else style)) as demo:
    gr.HTML(HEADER)
    with gr.Row(equal_height=False):
        with gr.Column(scale=4):
            inp = gr.Textbox(label='Comment (Banglish)', lines=4,
                             placeholder='Type or paste a comment in Bangla written with English letters...')
            with gr.Row():
                method = gr.Radio(['SHAP', 'LIME'], value='SHAP', label='Explanation method', scale=2)
                btn = gr.Button('Analyze', variant='primary', size='lg', scale=1)
            examples = gr.Dataset(components=[inp], samples=[[e] for e in EXAMPLES],
                                  label='Try an example (click to analyze)', samples_per_page=10)
            gr.HTML(BIAS_NOTE)
            bias_examples = gr.Dataset(components=[inp], samples=[[e] for e in BIAS_EXAMPLES],
                                       label='Known weakness: harmless sentences wrongly flagged', samples_per_page=10)
        with gr.Column(scale=6):
            verdict = gr.HTML(EMPTY_VERDICT)
            why = gr.HTML(EMPTY_WHY)
            cats = gr.HTML('')
    with gr.Accordion('How it works', open=False):
        gr.Markdown(ABOUT)
    gr.HTML(FOOTER)

    btn.click(analyze, [inp, method], [verdict, cats, why])
    examples.click(make_picker(EXAMPLES), [examples, method], [inp, verdict, cats, why])
    bias_examples.click(make_picker(BIAS_EXAMPLES), [bias_examples, method], [inp, verdict, cats, why])
    method.change(reexplain, [inp, method], [why])

## Launch

share=True gives a public link (ending in gradio.live) that works on any device while this cell is running. The app also shows up inside Colab below the cell.

For the show: start this notebook about 10 minutes before, open the link on the projector laptop, and press Analyze once so the model is warmed up.

In [12]:
demo.queue().launch(share=True, show_error=True, **(style if GR6 else {}))

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f61c4b1437ff860f4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Screenshots for the report

Take 3 screenshots of the app (Windows: Win + Shift + S) and save them in results/figures with these names:
1. fig_09_demo_hate.png: a hate example with SHAP, showing the verdict, categories and highlighted words,
2. fig_09_demo_not_hate.png: a normal comment predicted as not hate,
3. fig_09_demo_lime.png: the same comment as in screenshot 1, switched to LIME.

Crop to the app window and keep the browser zoom at 100%. For the report, open the link with ?__theme=light at the end so the screenshots have a white background, which prints better.

## Remember this

- What we did: built a web app that shows the hate label, the hate categories and the words behind each decision, and runs from Colab with a public link.
- Why: moderators need a tool they can use and trust, not just a notebook, and the highlighted words show why a comment was flagged.
- Key number: one click gives a verdict, 8 category scores and a word-level explanation in about 1 to 3 seconds on a T4 GPU.